# Real Training with Real Data (GEOshield)

This Google Colab notebook trains an XGBoost model to predict Electron Flux using the GOES 11-year historical dataset. Ensure you have the `data/goes_historical_features.parquet` file uploaded to your environment or accessible in your Google Drive.

In [ ]:
# Install dependencies if running in Colab
!pip install xgboost scikit-learn pandas numpy matplotlib pyarrow fastparquet

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import json
import os

## 1. Load Data
Loading 11-year historical dataset. If you mounted Google Drive, adjust the path to point to your `goes_historical_features.parquet` file.

In [ ]:
# Adjust path as needed for Google Colab/Drive
data_path = '../data/goes_historical_features.parquet'

try:
    df = pd.read_parquet(data_path)
    df.sort_values('timestamp', inplace=True)
    df['year'] = df['timestamp'].dt.year
    print("Data loaded successfully! Shape:", df.shape)
    display(df.head())
except Exception as e:
    print(f"Could not load data at {data_path}. Exception: {e}")
    print("Please update data_path to point to the correct parquet file.")

## 2. Prepare Data for XGBoost

In [ ]:
p95_val = df['Electron_Flux'].quantile(0.95)
p99_val = df['Electron_Flux'].quantile(0.99)

train_mask = (df['year'] >= 2010) & (df['year'] <= 2017)
valid_mask = (df['year'] >= 2018) & (df['year'] <= 2019)

target_col = 'Target_12h'
features = [c for c in df.columns if c not in ['timestamp', 'year', 'Target_45m', 'Target_6h', 'Target_12h', 'Proton_Flux']]
valid_rows = df[target_col].notna()

tr_df = df[train_mask & valid_rows].copy()
tr_y_raw = tr_df[target_col]

weights = np.ones(len(tr_df))
weights[tr_y_raw > p95_val] *= 5
weights[tr_y_raw > p99_val] *= 15
tr_df['_weight'] = weights

storm_mask = tr_y_raw > p95_val
quiet_mask = ~storm_mask
num_storms = storm_mask.sum()
target_quiet = num_storms * 4

if num_storms > 0 and target_quiet < quiet_mask.sum():
    quiet_idx = tr_df[quiet_mask].sample(n=target_quiet, random_state=42).index
    final_train_idx = quiet_idx.union(tr_df[storm_mask].index)
else:
    final_train_idx = tr_df.index

X_tr = tr_df.loc[final_train_idx][features]
y_tr = np.log10(tr_df.loc[final_train_idx][target_col] + 1)
w_tr = tr_df.loc[final_train_idx]['_weight']

X_va = df[valid_mask & valid_rows][features]
y_va_raw = df[valid_mask & valid_rows][target_col]
y_va_base = df[valid_mask & valid_rows]['Electron_Flux']

print(f"Training set shape: {X_tr.shape}")
print(f"Validation set shape: {X_va.shape}")

## 3. Train the Model

In [ ]:
xgb_params = {
    'objective': 'reg:squarederror',
    'max_depth': 10,
    'learning_rate': 0.02,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'min_child_weight': 20,
    'gamma': 0.5,
    'reg_alpha': 0.2,
    'reg_lambda': 2,
    'n_estimators': 200, # Reduced for faster demo
    'n_jobs': -1
}

print(f"Pretraining on {len(X_tr)} samples... (12h Horizon)")
model = xgb.XGBRegressor(**xgb_params)
model.fit(X_tr, y_tr, sample_weight=w_tr, verbose=True)

## 4. Evaluate the Model

In [ ]:
def calc_metrics(y_true, y_pred, p95_val, p99_val, y_true_base=None):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    true_95 = y_true > p95_val
    pred_95 = y_pred > p95_val
    recall_95 = np.sum(true_95 & pred_95) / np.sum(true_95) if np.sum(true_95) > 0 else 0
    true_99 = y_true > p99_val
    pred_99 = y_pred > p99_val
    recall_99 = np.sum(true_99 & pred_99) / np.sum(true_99) if np.sum(true_99) > 0 else 0
    metrics = {
        'RMSE': float(rmse),
        'PeakRecall95': float(recall_95),
        'PeakRecall99': float(recall_99)
    }
    if y_true_base is not None:
        rmse_base = np.sqrt(mean_squared_error(y_true, y_true_base))
        metrics['Imp_vs_Persist_RMSE'] = float(100 * (rmse_base - rmse) / rmse_base)
    return metrics

y_va_pred_log = model.predict(X_va)
y_va_pred_raw = np.power(10, y_va_pred_log) - 1
y_va_pred_raw = np.clip(y_va_pred_raw, 0, None)

mets = calc_metrics(y_va_raw, y_va_pred_raw, p95_val, p99_val, y_va_base)
print("Validation Metrics:", json.dumps(mets, indent=2))

## 5. Feature Importance

In [ ]:
imp_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(imp_df['Feature'].head(15)[::-1], imp_df['Importance'].head(15)[::-1])
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()